# Automated Code Review Board — End-to-End Multi-Agent Demo
**Course:** CSE476: Agentic AI and Intelligent Automation (Group 14)

This notebook demonstrates the complete Semantic Kernel Fan-Out / Fan-In workflow:
1. Initializes the Semantic Kernel environment and `SharedChangeState`.
2. Submits a sample code change (a Git diff with hardcoded secrets, SQL injection, `eval()`, and missing tests).
3. Executes the 4 parallel specialist agents (`DiffAgent`, `StyleAgent`, `SecurityAgent`, `TestCoverageAgent`), followed by `ReviewerManager` and `FixSuggestionAgent`.
4. Prints the execution trace, per-agent findings, consolidated merge report, and the validated unified diff patch.

In [1]:
import asyncio
import json
import sys
from pathlib import Path

# Add backend directory to sys.path
BACKEND_DIR = Path.cwd() / 'backend' if (Path.cwd() / 'backend').exists() else Path.cwd().parent / 'backend'
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from orchestration.schemas import SubmissionMetadata
from orchestration.state import SharedChangeState
from orchestration.workflow import execute_review_workflow
from services.llm_service import create_kernel
from services.report_service import append_execution_event

kernel, is_demo, model_id = create_kernel(demo_mode=True)
print(f'Semantic Kernel initialized | demo_mode={is_demo} | model={model_id}')

Semantic Kernel initialized | demo_mode=True | model=demo-deterministic-groq-simulator


In [2]:
SAMPLE_GIT_DIFF = '''diff --git a/payments/checkout.py b/payments/checkout.py
new file mode 100644
--- /dev/null
+++ b/payments/checkout.py
@@ -0,0 +1,15 @@
+import sqlite3
+
+STRIPE_SECRET = "sk-live-1234567890abcdef1234567890"
+
+def lookup_order(conn, order_id):
+    cur = conn.cursor()
+    cur.execute(f"SELECT * FROM orders WHERE id = '{order_id}'")
+    return cur.fetchone()
+
+def parse_discount_rule(rule_expr):
+    return eval(rule_expr)
'''

state = SharedChangeState(
    submission_metadata=SubmissionMetadata(
        title='Checkout & Discount Rule Patch',
        description='Adds order lookup and dynamic discount expression parsing.',
        submission_type='diff',
        demo_mode=True,
    ),
    files=[],
    git_diff=SAMPLE_GIT_DIFF,
    event_sink=append_execution_event,
)
print('Created SharedChangeState review_id:', state.review_id)

Created SharedChangeState review_id: e5064a48-be9d-4a52-b12b-d4e38c5c7d51


In [3]:
# Execute the complete Semantic Kernel Fan-Out / Fan-In workflow
result_state = await execute_review_workflow(state, demo_mode=True)
print('Workflow status:', result_state.data.overall_status)
print('Total duration (ms):', result_state.data.execution_timestamps.total_duration_ms)

Workflow status: completed
Total duration (ms): 49385


In [4]:
print('=== EXECUTION TRACE ===')
for ev in result_state.data.events:
    print(f'[{ev.timestamp}] {ev.agent_name:22s} | {ev.event:18s} | {ev.status:9s} | {ev.message}')

=== EXECUTION TRACE ===
[2026-11-12T08:22:38.638911+00:00] reviewer_manager       | workflow_started   | running   | Reviewer Manager initialized shared change state and started fan-out.
[2026-11-12T08:22:38.646496+00:00] reviewer_manager       | fan_out_started    | running   | Dispatched Diff, Style, Security, and Test-Coverage agents in parallel.
[2026-11-12T08:22:38.658875+00:00] diff_agent             | agent_started      | running   | Parsing Git diff and extracting modified files...
[2026-11-12T08:22:38.660163+00:00] diff_agent             | agent_progress     | running   | Running parse_git_diff & extract_modified_files
[2026-11-12T08:22:38.663165+00:00] diff_agent             | agent_progress     | running   | Running analyze_code_structure
[2026-11-12T08:22:38.665162+00:00] diff_agent             | agent_progress     | running   | Synthesizing change summary with Semantic Kernel agent...
[2026-11-12T08:22:38.666165+00:00] style_agent            | agent_started      | running 

In [5]:
print('=== AGENT FINDINGS & TOOLS ===')
for agent_name, status in result_state.data.agent_statuses.items():
    findings = result_state.data.agent_findings.get(agent_name, [])
    print(f'\n--- {status.display_name} ({status.status.value}) | tools={status.tools_invoked} ---')
    print('Summary:', status.summary)
    for f in findings:
        print(f'  [{f.severity.value}] {f.title} ({f.file_path}:{f.line_start}) -> {f.evidence}')

=== AGENT FINDINGS & TOOLS ===

--- Diff Agent (completed) | tools=['parse_git_diff', 'analyze_code_structure'] ---
Summary: [Demo AI Analysis] Reviewed 1 file(s) with 2 extracted symbol(s) and 2 potentially risky structural touch(es).
  [INFO] Risky structural touch: Database query execution (payments/checkout.py:7) -> cur.execute(f"SELECT * FROM orders WHERE id = '{order_id}'")
  [INFO] Risky structural touch: Dynamic code execution (payments/checkout.py:11) -> return eval(rule_expr)

--- Style Agent (completed) | tools=['run_linter', 'calculate_complexity'] ---
Summary: [Demo AI Analysis] Interpreted 0 lint issue(s), 0 complexity warning(s), and 0 duplication block(s).

--- Security Agent (completed) | tools=['scan_for_secrets', 'run_security_analysis'] ---
Summary: [Demo AI Analysis] Evaluated 1 secret detection(s) and 4 static security pattern finding(s).
  [CRITICAL] Hardcoded OpenAI / Bearer API Key (payments/checkout.py:3) -> STRIPE_SECRET = "sk-************890"
  [CRITICAL] Po

In [6]:
rec = result_state.data.merge_recommendation
print('=== CONSOLIDATED REVIEW REPORT ===')
print('Merge Decision:', rec.decision.value)
print('Reasoning:', rec.reasoning)
print('Severity Summary:', json.dumps(result_state.data.severity_summary))
print('\nRanked Findings:')
for idx, f in enumerate(result_state.data.consolidated_findings, start=1):
    print(f'  {idx}. [{f.severity.value}] ({f.agent}) {f.title} @ {f.file_path}:{f.line_start}')

=== CONSOLIDATED REVIEW REPORT ===
Merge Decision: REQUEST_CHANGES
Reasoning: Merge recommendation is REQUEST_CHANGES. 4 CRITICAL finding(s) detected that block merge.
Severity Summary: {"CRITICAL": 4, "HIGH": 0, "MEDIUM": 1, "LOW": 0, "INFO": 2}

Ranked Findings:
  1. [CRITICAL] (security_agent) Hardcoded OpenAI / Bearer API Key @ payments/checkout.py:3
  2. [CRITICAL] (security_agent) Potential SQL Injection via dynamic string interpolation @ payments/checkout.py:7
  3. [CRITICAL] (security_agent) Parameterized query violation in cursor.execute() @ payments/checkout.py:7
  4. [CRITICAL] (security_agent) Unsafe dynamic code execution (eval / exec) @ payments/checkout.py:11
  5. [MEDIUM] (test_coverage_agent) No test files submitted or mapped to modified code @ payments/checkout.py:1
  6. [INFO] (diff_agent) Risky structural touch: Database query execution @ payments/checkout.py:7
  7. [INFO] (diff_agent) Risky structural touch: Dynamic code execution @ payments/checkout.py:11


In [7]:
fix = result_state.data.fix_suggestion
print('=== FIX-SUGGESTION AGENT PROPOSED PATCH ===')
print('Available:', fix.available)
print('Validation Status:', fix.validation_status, '|', fix.validation_message)
print('Selected Finding:', fix.selected_finding.title if fix.selected_finding else None)
print('Explanation:', fix.explanation)
print('Requires Explicit User Approval:', fix.requires_explicit_approval, '| Approved:', fix.user_approved)
print('\nOriginal Code:\n' + fix.original_code)
print('\nSuggested Replacement:\n' + fix.suggested_replacement)
print('\nUnified Diff Patch:\n' + fix.proposed_patch)

=== FIX-SUGGESTION AGENT PROPOSED PATCH ===
Available: True
Validation Status: valid | Unified diff format and static syntax validation passed (not executed).
Selected Finding: Hardcoded OpenAI / Bearer API Key
Explanation: Reads `STRIPE_SECRET` from environment variable `STRIPE_SECRET` instead of hardcoding a credential in source code.
Requires Explicit User Approval: True | Approved: False

Original Code:
STRIPE_SECRET = "sk-************890"

Suggested Replacement:
STRIPE_SECRET = os.getenv("STRIPE_SECRET", "")

Unified Diff Patch:
--- a/payments/checkout.py
+++ b/payments/checkout.py
@@ -1,6 +1,6 @@
 import sqlite3
 
-STRIPE_SECRET = "sk-************890"
+STRIPE_SECRET = os.getenv("STRIPE_SECRET", "")
 
 def lookup_order(conn, order_id):
     cur = conn.cursor()

